## Langgraph tool integration


In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode , tools_condition
from langchain_tavily import TavilySearch
from langchain_core.tools import Tool

import requests
import math


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)

In [4]:
import os
import math
import requests

from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

# ---------------------------------------
# Load Environment Variables
# ---------------------------------------

load_dotenv()

# Read API Keys
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
ALPHA_VANTAGE_API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")

# Check if keys are loaded
if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY not found!")

if not ALPHA_VANTAGE_API_KEY:
    raise ValueError("ALPHA_VANTAGE_API_KEY not found!")

# ---------------------------------------
# Tavily Search Tool
# ---------------------------------------

search_tool = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced",
    tavily_api_key=TAVILY_API_KEY
)

# ---------------------------------------
# Calculator Tool
# ---------------------------------------

@tool
def calculator(expression: str) -> str:
    """
    Useful for simple mathematical calculations.

    Examples:
    - 2 + 2
    - math.sqrt(16)
    - 10 * 5
    """

    try:
        allowed = {
            "math": math,
            "abs": abs,
            "round": round,
            "min": min,
            "max": max,
            "sum": sum,
        }

        result = eval(expression, {"__builtins__": {}}, allowed)
        return str(result)

    except Exception as e:
        return f"Error: {e}"

# ---------------------------------------
# Stock Price Tool
# ---------------------------------------

@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch the latest stock price.

    Example:
    - AAPL
    - TSLA
    - MSFT
    """

    url = (
        "https://www.alphavantage.co/query"
        f"?function=GLOBAL_QUOTE"
        f"&symbol={symbol}"
        f"&apikey={ALPHA_VANTAGE_API_KEY}"
    )

    response = requests.get(url)

    return response.json()

# ---------------------------------------
# Test
# ---------------------------------------

print("Tavily Key Loaded :", TAVILY_API_KEY[:10] + "...")
print("Alpha Key Loaded  :", ALPHA_VANTAGE_API_KEY[:5] + "...")

Tavily Key Loaded : tvly-dev-4...
Alpha Key Loaded  : 5FYCY...


In [5]:
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")

if not WEATHER_API_KEY:
    raise ValueError("WEATHER_API_KEY not found!")

In [6]:
# weather tool
@tool
def get_weather(city: str) -> dict:
    """
    Get the current weather for a city.

    Args:
        city: Name of the city.

    Examples:
        Mumbai
        London
        New York
    """

    url = (
        f"http://api.weatherapi.com/v1/current.json"
        f"?key={WEATHER_API_KEY}"
        f"&q={city}"
        f"&aqi=no"
    )

    response = requests.get(url)

    if response.status_code != 200:
        return {
            "error": "Unable to fetch weather.",
            "details": response.json()
        }

    data = response.json()

    return {
        "city": data["location"]["name"],
        "country": data["location"]["country"],
        "temperature_c": data["current"]["temp_c"],
        "temperature_f": data["current"]["temp_f"],
        "condition": data["current"]["condition"]["text"],
        "humidity": data["current"]["humidity"],
        "wind_kph": data["current"]["wind_kph"],
        "feels_like_c": data["current"]["feelslike_c"],
        "last_updated": data["current"]["last_updated"]
    }

In [ ]:
#Making Tool List